# 01: Explore the T-7 renewal table (seed 42)

Exploration only. The notebook imports the installed package; train and serve code lives in `src/retention_radar/`.

Run `CHURN_DATA_SOURCE=synthetic ./scripts/run_all.sh` (or `python -m retention_radar.cli.generate_data`) first so `data/raw/renewals_t7.csv` and `data/raw/renewals_all.csv` exist.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from retention_radar import config
from retention_radar.serving.packet import build_decision_packet

t7 = pd.read_csv(config.RAW_DIR / "renewals_t7.csv")
all_renewals = pd.read_csv(config.RENEWALS_ALL_CSV)
print("model table:", t7.shape, " all renewals:", all_renewals.shape)

## Outcomes and routes

Every generated renewal is in `renewals_all.csv`. Failed cards go to dunning and scheduled cancels go to the cancel flow; only `route == "model"` rows reach the model table.

In [ ]:
pd.crosstab(all_renewals["outcome"], all_renewals["route"], margins=True)

In [ ]:
print(f"voluntary-lapse rate in the model table: {t7['churned'].mean():.3f}")

## Lapse rate by tenure and by cap hits

In [ ]:
tenure = pd.cut(t7["renewals_completed"], bins=[-1, 0, 2, 5, 11, 60],
                labels=["0 (first)", "1-2", "3-5", "6-11", "12+"])
t7.groupby(tenure, observed=True)["churned"].agg(["mean", "size"]).round(3)

In [ ]:
hits = t7["limit_hits_14d"].clip(upper=5).rename("limit_hits_14d (5 = 5+)")
t7.groupby(hits)["churned"].agg(["mean", "size"]).round(3)

## Score the worked examples

Maya and Arjun are scoring-time records (no label). The packet uses the committed `models/` bundle. Expected: Maya calibrated about 0.153, medium, `limit_reset`; Arjun about 0.023, low, `no_action`.

In [ ]:
for name, filename in config.HEROES.items():
    record = json.loads((config.HERO_DIR / filename).read_text())
    packet = build_decision_packet(record)
    s, d = packet["scoring"], packet["decision"]
    print(f"{name:6s} raw={s['churn_probability_raw']:.3f} "
          f"cal={s['churn_probability_calibrated']:.3f} band={s['risk_band']} "
          f"action={d['action']} auto_action={d['auto_action']}")

Published write-ups: `results/benchmarks.md`, `results/WORKED_EXAMPLES.md`. Charts: `results/plots/`.